# 🌍 India AQI Prediction Model
### Machine Learning for Air Quality Forecasting

This notebook predicts Air Quality Index (AQI) for major Indian cities using Random Forest models.

**Features:**
- 1-hour and 24-hour AQI predictions
- Feature importance analysis
- Model accuracy visualization
- Interactive predictions

**Cities covered:** Delhi, Mumbai, Kolkata, Hyderabad

## 📦 Step 1: Install Required Packages

In [1]:
# Install required packages (uncomment if needed)
# !pip install pandas matplotlib seaborn scikit-learn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

print("✅ All packages imported successfully!")

✅ All packages imported successfully!


## 📁 Step 2: Upload Your Dataset

**Choose ONE of the following options:**

### Option 1: Upload from Google Drive (Recommended)

If your dataset is on Google Drive, this is the easiest method!

In [2]:
# Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# INSTRUCTIONS:
# 1. Upload 'india_aqi_v6_2024-2025.csv' to your Google Drive
# 2. Update the path below to match your file location
# 3. Run this cell

# Example paths (uncomment and modify one):
# DRIVE_FILE_PATH = '/content/drive/MyDrive/india_aqi_v6_2024-2025.csv'
# DRIVE_FILE_PATH = '/content/drive/MyDrive/Datasets/india_aqi_v6_2024-2025.csv'

# After uploading to Drive, you can find the path by:
# - Right-click the file in Drive → Get link → Copy the file ID
# - Or browse using the folder icon on the left sidebar

print("✅ Google Drive mounted successfully!")
print("\n💡 TIP: Browse your Drive files using the folder icon (📁) on the left sidebar")
print("    Navigate to 'drive' → 'MyDrive' to find your files")

ModuleNotFoundError: No module named 'google'

### Option 2: Direct Upload Widget

Upload the CSV file directly from your computer.

In [1]:
from google.colab import files
import io

print("📤 Click 'Choose Files' and select 'india_aqi_v6_2024-2025.csv'")
print("⏳ Upload may take a moment depending on file size...\n")

uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f"\n✅ File '{filename}' uploaded successfully!")
    print(f"📊 File size: {len(uploaded[filename]) / 1024:.2f} KB")
else:
    print("❌ No file uploaded. Please try again.")

ModuleNotFoundError: No module named 'google'

### Option 3: Load from URL

If your dataset is hosted online (GitHub, Dropbox, etc.).

In [ ]:
import urllib.request

# Replace this with your dataset URL
# DATASET_URL = 'https://raw.githubusercontent.com/your-repo/india_aqi_v6_2024-2025.csv'
# Or use a direct Dropbox/Google Drive share link

# Uncomment and modify:
# print("⏳ Downloading dataset from URL...")
# urllib.request.urlretrieve(DATASET_URL, 'india_aqi_v6_2024-2025.csv')
# print("✅ Dataset downloaded successfully!")

print("💡 To use this option:")
print("   1. Host your CSV file online (GitHub, Dropbox, etc.)")
print("   2. Get the direct download link")
print("   3. Uncomment and update DATASET_URL above")
print("   4. Run this cell")

## 🔄 Step 3: Load and Clean Data

This cell will automatically find and load your dataset.

In [ ]:
import os
import glob

print("🔍 Searching for dataset...\n")

# Try to find the dataset file in multiple locations
possible_paths = [
    'india_aqi_v6_2024-2025.csv',  # Current directory
    '/content/india_aqi_v6_2024-2025.csv',  # Colab default
    '/content/drive/MyDrive/india_aqi_v6_2024-2025.csv',  # Google Drive root
]

# Also search in Drive subfolders if mounted
if os.path.exists('/content/drive/MyDrive'):
    drive_files = glob.glob('/content/drive/MyDrive/**/india_aqi_v6_2024-2025.csv', recursive=True)
    possible_paths.extend(drive_files)

# Find the file
dataset_path = None
for path in possible_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path:
    print(f"✅ Found dataset at: {dataset_path}\n")
    print("1. Loading and cleaning data...")
    
    # Load Data
    df = pd.read_csv(dataset_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values(by=['city', 'datetime'])
    
    print(f"✅ Loaded {len(df):,} records")
    print(f"📅 Date range: {df['datetime'].min()} to {df['datetime'].max()}")
    print(f"🏙️  Cities: {', '.join(df['city'].unique())}")
    print(f"\n📊 Dataset Preview:")
    display(df.head())
else:
    print("❌ Dataset not found!\n")
    print("📋 Please do ONE of the following:")
    print("   1. Run one of the upload cells above (Step 2)")
    print("   2. Make sure the file is named 'india_aqi_v6_2024-2025.csv'")
    print("   3. If using Google Drive, ensure it's mounted and the file path is correct")
    print("\n💡 After uploading, run this cell again.")
    raise FileNotFoundError("Dataset file 'india_aqi_v6_2024-2025.csv' not found")

## ⚙️ Step 4: Feature Engineering

In [ ]:
print("2. Feature Engineering...")

# Create Lag Features (Looking at the past)
df['AQI_lag1'] = df.groupby('city')['AQI'].shift(1)
df['AQI_lag24'] = df.groupby('city')['AQI'].shift(24)

# Create Targets (What we are trying to predict)
df['Target_Next_1h_AQI'] = df.groupby('city')['AQI'].shift(-1)
df['Target_Next_24h_AQI'] = df.groupby('city')['AQI'].shift(-24)

# Drop missing values caused by shifting, and One-Hot Encode Cities
df = df.dropna(subset=['Target_Next_1h_AQI', 'Target_Next_24h_AQI', 'AQI_lag1', 'AQI_lag24'])
df_encoded = pd.get_dummies(df, columns=['city'], drop_first=False)

# Define our highly accurate feature set
features = [
    'AQI', 'Temperature_C', 'Precipitation_mm', 'WindSpeed_kmh', 
    'hour', 'month', 'AQI_lag1', 'AQI_lag24',
    'city_Delhi', 'city_Hyderabad', 'city_Kolkata', 'city_Mumbai'
]

X = df_encoded[features]
y_1h = df_encoded['Target_Next_1h_AQI']
y_24h = df_encoded['Target_Next_24h_AQI']

print(f"✅ Created {len(features)} features")
print(f"📊 Training samples: {len(X):,}")
print(f"\nFeatures: {', '.join(features)}")

## 🔀 Step 5: Train-Test Split

In [ ]:
# Train-Test Split (80% for training, 20% for testing the accuracy)
X_train, X_test, y_1h_train, y_1h_test = train_test_split(
    X, y_1h, test_size=0.2, random_state=42, shuffle=False
)
_, _, y_24h_train, y_24h_test = train_test_split(
    X, y_24h, test_size=0.2, random_state=42, shuffle=False
)

print(f"✅ Data split complete:")
print(f"   Training set: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"   Test set: {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")

## 🤖 Step 6: Train Random Forest Models

This will train two models:
- **1-Hour Prediction Model**: Predicts AQI for the next hour
- **24-Hour Prediction Model**: Predicts AQI for 24 hours ahead

In [ ]:
print("3. Training Random Forest Models (This may take a minute)...")

# Train 1-hour model
rf_1h = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_1h.fit(X_train, y_1h_train)
print("✅ 1-hour model trained")

# Train 24-hour model
rf_24h = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_24h.fit(X_train, y_24h_train)
print("✅ 24-hour model trained")

print("\n🎉 Both models trained successfully!")

## 📈 Step 7: Model Accuracy Evaluation

In [ ]:
print("--- MODEL ACCURACY EVALUATION ---\n")

# Predict on the test set
preds_1h = rf_1h.predict(X_test)
preds_24h = rf_24h.predict(X_test)

# Calculate Accuracy Metrics
r2_1h = r2_score(y_1h_test, preds_1h)
mae_1h = mean_absolute_error(y_1h_test, preds_1h)
r2_24h = r2_score(y_24h_test, preds_24h)
mae_24h = mean_absolute_error(y_24h_test, preds_24h)

print(f"🎯 1-Hour Model:")
print(f"   R² Score: {r2_1h:.4f} ({r2_1h*100:.2f}% variance explained)")
print(f"   Mean Absolute Error: ±{mae_1h:.2f} AQI points")
print(f"\n🎯 24-Hour Model:")
print(f"   R² Score: {r2_24h:.4f} ({r2_24h*100:.2f}% variance explained)")
print(f"   Mean Absolute Error: ±{mae_24h:.2f} AQI points")

# Display interpretation
print("\n📊 What this means:")
if r2_1h > 0.9:
    print("   ✅ Excellent accuracy for 1-hour predictions!")
elif r2_1h > 0.7:
    print("   ✅ Good accuracy for 1-hour predictions")
else:
    print("   ⚠️  Moderate accuracy for 1-hour predictions")

if r2_24h > 0.7:
    print("   ✅ Strong accuracy for 24-hour predictions!")
elif r2_24h > 0.5:
    print("   ✅ Decent accuracy for 24-hour predictions")
else:
    print("   ⚠️  Moderate accuracy for 24-hour predictions")

## 📊 Step 8: Visualizations

In [ ]:
# Set chart style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

### 🔍 Chart 1: Feature Importance
Which factors matter most for AQI prediction?

In [ ]:
plt.figure(figsize=(10, 6))
importances = rf_1h.feature_importances_
feat_df = pd.DataFrame({
    'Feature': features, 
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

sns.barplot(x='Importance', y='Feature', data=feat_df, palette='viridis')
plt.title('What Drives AQI? (Feature Importance)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Top 5 Most Important Features:")
for idx, row in feat_df.head().iterrows():
    print(f"   {row['Feature']}: {row['Importance']:.4f}")

### 📉 Chart 2: Actual vs Predicted (Line Plot)
How well does the model track real AQI changes?

In [ ]:
plt.figure(figsize=(14, 5))
sample_size = min(100, len(y_1h_test))

plt.plot(range(sample_size), y_1h_test.values[:sample_size], 
         label='Actual AQI', marker='o', linestyle='-', alpha=0.7, color='blue', markersize=4)
plt.plot(range(sample_size), preds_1h[:sample_size], 
         label='Predicted AQI', marker='x', linestyle='--', alpha=0.9, color='red', markersize=4)

plt.title('Tracking Accuracy: Actual vs Predicted AQI (Sample Window)', fontsize=14, fontweight='bold')
plt.xlabel('Time (Hours into Test Set)', fontsize=12)
plt.ylabel('AQI', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('actual_vs_predicted_line.png', dpi=300, bbox_inches='tight')
plt.show()

### 🎯 Chart 3: 24-Hour Prediction Scatter Plot
Points closer to the diagonal line = better predictions

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_24h_test, preds_24h, alpha=0.3, color='coral', s=20)
plt.plot([y_24h_test.min(), y_24h_test.max()], 
         [y_24h_test.min(), y_24h_test.max()], 
         'k--', lw=2, label='Perfect Prediction Line')

plt.title('24-Hour Forecast Accuracy Scatter Plot', fontsize=14, fontweight='bold')
plt.xlabel('Actual AQI', fontsize=12)
plt.ylabel('Predicted AQI', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('actual_vs_predicted_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

## 🔮 Step 9: Make Custom Predictions

Enter your own conditions to predict future AQI!

In [ ]:
# Function to format data and predict
def predict_future_aqi(city, current_aqi, aqi_1h_ago, aqi_24h_ago, temp, precip, wind, hour, month):
    """
    Predict AQI for 1 hour and 24 hours ahead.
    
    Parameters:
    - city: One of 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai'
    - current_aqi: Current AQI value
    - aqi_1h_ago: AQI from 1 hour ago
    - aqi_24h_ago: AQI from 24 hours ago
    - temp: Temperature in Celsius
    - precip: Precipitation in mm
    - wind: Wind speed in km/h
    - hour: Hour of day (0-23)
    - month: Month (1-12)
    """
    input_data = {
        'AQI': [current_aqi], 
        'Temperature_C': [temp], 
        'Precipitation_mm': [precip],
        'WindSpeed_kmh': [wind], 
        'hour': [hour], 
        'month': [month],
        'AQI_lag1': [aqi_1h_ago], 
        'AQI_lag24': [aqi_24h_ago],
        'city_Delhi': [1 if city == 'Delhi' else 0],
        'city_Hyderabad': [1 if city == 'Hyderabad' else 0],
        'city_Kolkata': [1 if city == 'Kolkata' else 0],
        'city_Mumbai': [1 if city == 'Mumbai' else 0]
    }
    input_df = pd.DataFrame(input_data)[features]
    return rf_1h.predict(input_df)[0], rf_24h.predict(input_df)[0]

print("✅ Prediction function ready!")

### 🎮 Interactive Prediction

Modify the values below and run the cell to get predictions!

In [ ]:
# ===== CUSTOMIZE THESE VALUES =====
city = 'Delhi'              # Options: 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai'
current_aqi = 85            # Current AQI
aqi_1h_ago = 82             # AQI 1 hour ago
aqi_24h_ago = 90            # AQI 24 hours ago
temperature = 30.0          # Temperature in °C
precipitation = 0.0         # Precipitation in mm
wind_speed = 15.5           # Wind speed in km/h
hour_of_day = 14            # Hour (0-23)
month_of_year = 3           # Month (1-12)
# ==================================

predicted_1h, predicted_24h = predict_future_aqi(
    city, current_aqi, aqi_1h_ago, aqi_24h_ago, 
    temperature, precipitation, wind_speed, hour_of_day, month_of_year
)

print("="*50)
print(f"🔮 PREDICTION FOR {city.upper()}")
print("="*50)
print(f"\n📍 Current Conditions:")
print(f"   Current AQI: {current_aqi}")
print(f"   Temperature: {temperature}°C")
print(f"   Wind Speed: {wind_speed} km/h")
print(f"   Precipitation: {precipitation} mm")
print(f"   Time: Hour {hour_of_day}, Month {month_of_year}")

print(f"\n🔮 Predictions:")
print(f"   Next 1 Hour:  {predicted_1h:.1f} AQI")
print(f"   Next 24 Hours: {predicted_24h:.1f} AQI")

# AQI Category
def get_aqi_category(aqi):
    if aqi <= 50:
        return "Good 🟢"
    elif aqi <= 100:
        return "Satisfactory 🟡"
    elif aqi <= 200:
        return "Moderate 🟠"
    elif aqi <= 300:
        return "Poor 🔴"
    elif aqi <= 400:
        return "Very Poor 🟣"
    else:
        return "Severe 🟤"

print(f"\n📊 Air Quality Category:")
print(f"   1-Hour: {get_aqi_category(predicted_1h)}")
print(f"   24-Hour: {get_aqi_category(predicted_24h)}")
print("="*50)

### 🔄 Batch Predictions (Multiple Scenarios)

Compare predictions across different scenarios!

In [ ]:
# Define multiple scenarios
scenarios = [
    {'name': 'Normal Day Delhi', 'city': 'Delhi', 'current_aqi': 85, 'aqi_1h_ago': 82, 'aqi_24h_ago': 90, 
     'temp': 30, 'precip': 0, 'wind': 15.5, 'hour': 14, 'month': 3},
    
    {'name': 'Rainy Day Mumbai', 'city': 'Mumbai', 'current_aqi': 60, 'aqi_1h_ago': 65, 'aqi_24h_ago': 70, 
     'temp': 28, 'precip': 10, 'wind': 20, 'hour': 10, 'month': 7},
    
    {'name': 'Winter Morning Kolkata', 'city': 'Kolkata', 'current_aqi': 150, 'aqi_1h_ago': 145, 'aqi_24h_ago': 140, 
     'temp': 18, 'precip': 0, 'wind': 8, 'hour': 7, 'month': 1},
    
    {'name': 'Summer Evening Hyderabad', 'city': 'Hyderabad', 'current_aqi': 95, 'aqi_1h_ago': 90, 'aqi_24h_ago': 85, 
     'temp': 35, 'precip': 0, 'wind': 12, 'hour': 18, 'month': 5},
]

results = []
for scenario in scenarios:
    pred_1h, pred_24h = predict_future_aqi(
        scenario['city'], scenario['current_aqi'], scenario['aqi_1h_ago'], 
        scenario['aqi_24h_ago'], scenario['temp'], scenario['precip'], 
        scenario['wind'], scenario['hour'], scenario['month']
    )
    results.append({
        'Scenario': scenario['name'],
        'City': scenario['city'],
        'Current AQI': scenario['current_aqi'],
        'Predicted 1h': f"{pred_1h:.1f}",
        'Predicted 24h': f"{pred_24h:.1f}"
    })

results_df = pd.DataFrame(results)
print("\n📊 Batch Prediction Results:\n")
print(results_df.to_string(index=False))

## 💾 Download Generated Charts

The charts have been saved. You can download them from the Files panel on the left.

In [ ]:
import os

print("\n📁 Generated files:")
chart_files = ['feature_importance.png', 'actual_vs_predicted_line.png', 'actual_vs_predicted_scatter.png']
for f in chart_files:
    if os.path.exists(f):
        print(f"   ✅ {f}")
    else:
        print(f"   ❌ {f} (not found)")

print("\n💡 To download: Right-click on files in the left panel → Download")

## 🎉 All Done!

**What you've accomplished:**
- ✅ Loaded and processed AQI data for 4 major Indian cities
- ✅ Built two Random Forest models (1-hour and 24-hour predictions)
- ✅ Evaluated model accuracy with R² scores and MAE
- ✅ Generated 3 visualization charts
- ✅ Created an interactive prediction tool

**Next Steps:**
- Experiment with different scenarios in the prediction cells
- Download the charts for your reports/presentations
- Try modifying model parameters (n_estimators, max_depth, etc.)
- Add more features or cities to improve predictions

---
*Happy predicting! 🌍✨*